<a href="https://colab.research.google.com/github/nguyenducminh2206/NLP-TampereUniversity/blob/main/Week_6/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Named Entity Recognition (NER) Exercise
---

In [8]:
%pip install transformers datasets torchao
%pip install 'accelerate>=0.26.0'

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 14.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: "'accelerate": Expected package name at the start of dependency specifier
    'accelerate
    ^


## Preprocessing

### Import libraries

In [1]:
import os
import datasets
import pandas as pd
from transformers import pipeline
from transformers import BertTokenizerFast
from transformers import DataCollatorForTokenClassification
from transformers import AutoModelForTokenClassification
from transformers import Trainer, TrainingArguments
import torch
from torchao.quantization import Int8DynamicActivationInt8WeightConfig, quantize_
import copy

c:\Users\minhn\anaconda3\envs\dl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu130 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0212 00:04:10.271000 34072 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


### Load data and preprocess the DataFrame

In [2]:
ner_df = pd.read_csv('NER_dataset.csv', encoding='latin-1')
# Fill missing data in 'Sentence #' column
ner_df['Sentence #'] = ner_df['Sentence #'].ffill()
# Clean unambiguous text value 'None'
ner_df['Word'] = ner_df['Word'].fillna('None')
# Create Tag_ID column
ner_tags = ner_df['Tag'].unique()
tag2id = {tag: i for i, tag in enumerate(ner_tags)}
ner_df['Tag_ID'] = [tag2id[tag] for tag in ner_df['Tag']]
# Group words, POS, tags and tag IDs into lists by sentences
group_ner_df = ner_df.groupby(by='Sentence #').agg({
    'Word': lambda word: [w for w in word],
    'POS': lambda pos: [p for p in pos],
    'Tag': lambda tag: [t for t in tag],
    'Tag_ID': lambda tag_id: [tid for tid in tag_id],
}).reset_index()

### Create datast from the DataFrame

In [ ]:
ner_dataset = datasets.Dataset.from_pandas(group_ner_df).train_test_split(test_size=0.15, train_size=0.85)
ner_dataset

DatasetDict({
    train: Dataset({
        features: ['Sentence #', 'Word', 'POS', 'Tag', 'Tag_ID'],
        num_rows: 40765
    })
    test: Dataset({
        features: ['Sentence #', 'Word', 'POS', 'Tag', 'Tag_ID'],
        num_rows: 7194
    })
})

In [6]:
ner_dataset['train'][0]

{'Sentence #': 'Sentence: 34089',
 'Word': ['In',
  '1997',
  ',',
  'in',
  'the',
  'second',
  'presidential',
  'race',
  ',',
  'Didier',
  'RATSIRAKA',
  ',',
  'the',
  'leader',
  'during',
  'the',
  '1970s',
  'and',
  '1980s',
  ',',
  'was',
  'returned',
  'to',
  'the',
  'presidency',
  '.'],
 'POS': ['IN',
  'CD',
  ',',
  'IN',
  'DT',
  'JJ',
  'JJ',
  'NN',
  ',',
  'NNP',
  'NNP',
  ',',
  'DT',
  'NN',
  'IN',
  'DT',
  'NNS',
  'CC',
  'NNS',
  ',',
  'VBD',
  'VBN',
  'TO',
  'DT',
  'NN',
  '.'],
 'Tag': ['O',
  'B-tim',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-per',
  'I-per',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-tim',
  'I-tim',
  'I-tim',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'Tag_ID': [0,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  3,
  10,
  0,
  0,
  0,
  0,
  0,
  7,
  12,
  12,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

### Tokenize the dataset

In [7]:
# Define tokenizer
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [8]:
def tokenize_and_align_labels(examples, label_all_tokens=True):
    tokenized_inputs = tokenizer(examples["Word"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["Tag_ID"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        # word_ids() => Return a list mapping the tokens
        # to their actual word in the initial sentence.
        # It Returns a list indicating the word corresponding to each token.
        previous_word_idx = None
        label_ids = []
        # Special tokens like `` and `<\s>` are originally mapped to None
        # We need to set the label to -100 so they are automatically ignored in the loss function.
        for word_idx in word_ids:
            if word_idx is None:
                # set –100 as the label for these special tokens
                label_ids.append(-100)
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            elif word_idx != previous_word_idx:
                # if current word_idx is != prev then its the most regular case
                # and add the corresponding token
                label_ids.append(label[word_idx])
            else:
                # to take care of sub-words which have the same word_idx
                # set -100 as well for them, but only if label_all_tokens == False
                label_ids.append(label[word_idx] if label_all_tokens else -100)
                # mask the subword representations after the first subword

            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [9]:
tokenized_dataset = ner_dataset.map(tokenize_and_align_labels, batched=True)

Map: 100%|██████████| 7194/7194 [00:00<00:00, 7792.52 examples/s]


In [10]:
tokenized_dataset['train'][0]

{'Sentence #': 'Sentence: 34089',
 'Word': ['In',
  '1997',
  ',',
  'in',
  'the',
  'second',
  'presidential',
  'race',
  ',',
  'Didier',
  'RATSIRAKA',
  ',',
  'the',
  'leader',
  'during',
  'the',
  '1970s',
  'and',
  '1980s',
  ',',
  'was',
  'returned',
  'to',
  'the',
  'presidency',
  '.'],
 'POS': ['IN',
  'CD',
  ',',
  'IN',
  'DT',
  'JJ',
  'JJ',
  'NN',
  ',',
  'NNP',
  'NNP',
  ',',
  'DT',
  'NN',
  'IN',
  'DT',
  'NNS',
  'CC',
  'NNS',
  ',',
  'VBD',
  'VBN',
  'TO',
  'DT',
  'NN',
  '.'],
 'Tag': ['O',
  'B-tim',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-per',
  'I-per',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-tim',
  'I-tim',
  'I-tim',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O'],
 'Tag_ID': [0,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  3,
  10,
  0,
  0,
  0,
  0,
  0,
  7,
  12,
  12,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'input_ids': [101,
  1999,
  2722,
  1010,
  1999,
  1996,
  2117,
  4883,
  2679,
  1010,
  2106,
  3771,
  11432,

## Training Loop

### Define the model and training arguements

In [11]:
# Model
model = AutoModelForTokenClassification.from_pretrained('bert-base-uncased', num_labels=len(ner_tags))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# Training Args
training_args = TrainingArguments(
    output_dir='./training_results',
#    evaluation_strategy = "epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
)

# Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer)


In [13]:
# Initialize the Trainer
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator
)
trainer.train()

Step,Training Loss
500,0.257200
1000,0.163600
1500,0.156500
2000,0.140300
2500,0.137400
3000,0.107900
3500,0.103000
4000,0.101700
4500,0.097800
5000,0.100200


TrainOutput(global_step=7644, training_loss=0.11247915525451373, metrics={'train_runtime': 582.4412, 'train_samples_per_second': 209.97, 'train_steps_per_second': 13.124, 'total_flos': 2783933052605880.0, 'train_loss': 0.11247915525451373, 'epoch': 3.0})

## Save and test model

In [14]:
model.save_pretrained('original_ner_model')
tokenizer.save_pretrained('ner_tokenizer')

('ner_tokenizer\\tokenizer_config.json',
 'ner_tokenizer\\special_tokens_map.json',
 'ner_tokenizer\\vocab.txt',
 'ner_tokenizer\\added_tokens.json',
 'ner_tokenizer\\tokenizer.json')

### Postprocessing

In [37]:
# Example texts
text = "Take a look inside the $5bn SoFi stadium in Los Angeles which will host the opening game of the World Cup in 2026."
text_2 = "Tariffs on goods from India would be lowered to 18%, from 25%, after Indian Prime Minister Narendra Modi agreed to stop buying Russian oil, Trump had said."

In [38]:
# Define NER pipeline
ner_model = pipeline("ner", model="original_ner_model", tokenizer="ner_tokenizer")

# Run the model
results = ner_model(text)
print(results)

results_2 = ner_model(text_2)
print(results_2)

Device set to use cuda:0


[{'entity': 'LABEL_0', 'score': np.float32(0.9999275), 'index': 1, 'word': 'take', 'start': 0, 'end': 4}, {'entity': 'LABEL_0', 'score': np.float32(0.99993944), 'index': 2, 'word': 'a', 'start': 5, 'end': 6}, {'entity': 'LABEL_0', 'score': np.float32(0.999938), 'index': 3, 'word': 'look', 'start': 7, 'end': 11}, {'entity': 'LABEL_0', 'score': np.float32(0.9999385), 'index': 4, 'word': 'inside', 'start': 12, 'end': 18}, {'entity': 'LABEL_0', 'score': np.float32(0.99993575), 'index': 5, 'word': 'the', 'start': 19, 'end': 22}, {'entity': 'LABEL_0', 'score': np.float32(0.9995809), 'index': 6, 'word': '$', 'start': 23, 'end': 24}, {'entity': 'LABEL_0', 'score': np.float32(0.9969304), 'index': 7, 'word': '5', 'start': 24, 'end': 25}, {'entity': 'LABEL_0', 'score': np.float32(0.97557837), 'index': 8, 'word': '##bn', 'start': 25, 'end': 27}, {'entity': 'LABEL_0', 'score': np.float32(0.78868574), 'index': 9, 'word': 'so', 'start': 28, 'end': 30}, {'entity': 'LABEL_0', 'score': np.float32(0.6519

In [24]:
label_mapping = {f"LABEL_{i}": label for i, label in enumerate(tag2id.keys())}
label_mapping

{'LABEL_0': 'O',
 'LABEL_1': 'B-geo',
 'LABEL_2': 'B-gpe',
 'LABEL_3': 'B-per',
 'LABEL_4': 'I-geo',
 'LABEL_5': 'B-org',
 'LABEL_6': 'I-org',
 'LABEL_7': 'B-tim',
 'LABEL_8': 'B-art',
 'LABEL_9': 'I-art',
 'LABEL_10': 'I-per',
 'LABEL_11': 'I-gpe',
 'LABEL_12': 'I-tim',
 'LABEL_13': 'B-nat',
 'LABEL_14': 'B-eve',
 'LABEL_15': 'I-eve',
 'LABEL_16': 'I-nat'}

In [25]:
# Convert to a better-to-read version
for result in results:
  if result['entity'] in label_mapping:
    result['entity'] = label_mapping[result['entity']]
print(results)

for result in results_2:
  if result['entity'] in label_mapping:
    result['entity'] = label_mapping[result['entity']]
print(results_2)

[{'entity': 'O', 'score': np.float32(0.9999275), 'index': 1, 'word': 'take', 'start': 0, 'end': 4}, {'entity': 'O', 'score': np.float32(0.99993944), 'index': 2, 'word': 'a', 'start': 5, 'end': 6}, {'entity': 'O', 'score': np.float32(0.999938), 'index': 3, 'word': 'look', 'start': 7, 'end': 11}, {'entity': 'O', 'score': np.float32(0.9999385), 'index': 4, 'word': 'inside', 'start': 12, 'end': 18}, {'entity': 'O', 'score': np.float32(0.99993575), 'index': 5, 'word': 'the', 'start': 19, 'end': 22}, {'entity': 'O', 'score': np.float32(0.9995809), 'index': 6, 'word': '$', 'start': 23, 'end': 24}, {'entity': 'O', 'score': np.float32(0.99693155), 'index': 7, 'word': '5', 'start': 24, 'end': 25}, {'entity': 'O', 'score': np.float32(0.9755823), 'index': 8, 'word': '##bn', 'start': 25, 'end': 27}, {'entity': 'O', 'score': np.float32(0.7890329), 'index': 9, 'word': 'so', 'start': 28, 'end': 30}, {'entity': 'O', 'score': np.float32(0.6525847), 'index': 10, 'word': '##fi', 'start': 30, 'end': 32}, {

In [26]:
def process_entity(results):
    combined_entities = {}
    current_entity = []
    current_label = None

    for result in results:
        if '-B' in result['entity']:
            if current_entity:
                combined_entities[' '.join(current_entity)] = current_label.split('-')[1]
                current_entity = []

                current_label = result['entity']
                current_entity.append(result['word'])
        elif 'I-' in result['entity'] and current_label and result['entity'].split('-')[1] == current_label.split('-')[1]:
            current_entity.append(result['word'])

        else:
            if current_entity:
                combined_entities[' '.join(current_entity)] = current_label.split('-')[1]
                current_entity = []

            current_label = result['entity'] if 'B-' in result['entity'] else None
            if current_label:
                current_entity.append(result['word'])
    if current_entity:
        combined_entities[' '.join(current_entity)] = current_label.split('-')[1]

    return combined_entities


In [27]:
process_entity(results)

{'los angeles': 'geo', 'world cup': 'org', '202': 'tim', '##6': 'tim'}

In [28]:
process_entity(results_2)

{'india': 'geo',
 'indian': 'gpe',
 'prime minister na ##ren ##dra mod ##i': 'per',
 'russian': 'gpe',
 'trump': 'per'}

## Model quantization

### Create a NER quantized model and compare model sizes

In [29]:
# Load pretrained model
model = AutoModelForTokenClassification.from_pretrained('original_ner_model')
tokenizer = BertTokenizerFast.from_pretrained('ner_tokenizer')

In [30]:
# Create a quantized model
model.eval()
quantized_model = copy.deepcopy(model)
quantize_(quantized_model, Int8DynamicActivationInt8WeightConfig())

In [ ]:
# Save models as pth files for size comparison
torch.save(model.state_dict(), "original_ner_model.pth")
torch.save(quantized_model.state_dict(), "quantized_ner_model.pth")

ner_model_size = os.path.getsize("original_ner_model.pth") / 1024**2
quantized_ner_model_size = os.path.getsize("quantized_ner_model.pth") / 1024**2
print(f"Original NER model size: {ner_model_size:.2f} MB")
print(f"Quantized NER model size: {quantized_ner_model_size:.2f} MB")

Original NER model size: 415.52 MB
Quantized NER model size: 172.84 MB


### Test the quantized model

In [39]:
# Define NER pipeline
ner_model_quantized = pipeline("ner", model=quantized_model, tokenizer=tokenizer)

# Run the model
q_results = ner_model_quantized(text)
print(q_results)

q_results_2 = ner_model_quantized(text_2)
print(q_results_2)

Device set to use cuda:0


[{'entity': 'LABEL_0', 'score': np.float32(0.99992514), 'index': 1, 'word': 'take', 'start': 0, 'end': 4}, {'entity': 'LABEL_0', 'score': np.float32(0.99993896), 'index': 2, 'word': 'a', 'start': 5, 'end': 6}, {'entity': 'LABEL_0', 'score': np.float32(0.99993694), 'index': 3, 'word': 'look', 'start': 7, 'end': 11}, {'entity': 'LABEL_0', 'score': np.float32(0.99993765), 'index': 4, 'word': 'inside', 'start': 12, 'end': 18}, {'entity': 'LABEL_0', 'score': np.float32(0.9999356), 'index': 5, 'word': 'the', 'start': 19, 'end': 22}, {'entity': 'LABEL_0', 'score': np.float32(0.9995616), 'index': 6, 'word': '$', 'start': 23, 'end': 24}, {'entity': 'LABEL_0', 'score': np.float32(0.99763215), 'index': 7, 'word': '5', 'start': 24, 'end': 25}, {'entity': 'LABEL_0', 'score': np.float32(0.9850513), 'index': 8, 'word': '##bn', 'start': 25, 'end': 27}, {'entity': 'LABEL_0', 'score': np.float32(0.8510882), 'index': 9, 'word': 'so', 'start': 28, 'end': 30}, {'entity': 'LABEL_0', 'score': np.float32(0.72

In [40]:
# Convert to a better-to-read version
for result in q_results:
  if result['entity'] in label_mapping:
    result['entity'] = label_mapping[result['entity']]
print(q_results)

for result in q_results_2:
  if result['entity'] in label_mapping:
    result['entity'] = label_mapping[result['entity']]
print(q_results_2)

[{'entity': 'O', 'score': np.float32(0.99992514), 'index': 1, 'word': 'take', 'start': 0, 'end': 4}, {'entity': 'O', 'score': np.float32(0.99993896), 'index': 2, 'word': 'a', 'start': 5, 'end': 6}, {'entity': 'O', 'score': np.float32(0.99993694), 'index': 3, 'word': 'look', 'start': 7, 'end': 11}, {'entity': 'O', 'score': np.float32(0.99993765), 'index': 4, 'word': 'inside', 'start': 12, 'end': 18}, {'entity': 'O', 'score': np.float32(0.9999356), 'index': 5, 'word': 'the', 'start': 19, 'end': 22}, {'entity': 'O', 'score': np.float32(0.9995616), 'index': 6, 'word': '$', 'start': 23, 'end': 24}, {'entity': 'O', 'score': np.float32(0.99763215), 'index': 7, 'word': '5', 'start': 24, 'end': 25}, {'entity': 'O', 'score': np.float32(0.9850513), 'index': 8, 'word': '##bn', 'start': 25, 'end': 27}, {'entity': 'O', 'score': np.float32(0.8510882), 'index': 9, 'word': 'so', 'start': 28, 'end': 30}, {'entity': 'O', 'score': np.float32(0.7233289), 'index': 10, 'word': '##fi', 'start': 30, 'end': 32}

In [41]:
process_entity(q_results)

{'los angeles': 'geo', 'world cup': 'org', '202': 'tim', '##6': 'tim'}

In [42]:
process_entity(q_results_2)

{'india': 'geo',
 'indian': 'gpe',
 'prime minister na ##ren ##dra mod ##i': 'per',
 'russian': 'gpe',
 'trump': 'per'}

## Comparison between original and quantized model

Dynamic quantization is implemented, which convert the fp32 original model to int8 quantized model.
- Model size: The quantized model size is less than a half of the original model size.
- Output result: The results are almost the same. There are only very little differences between the score values, so the token classification results from both models are still the same.